# T-339 — Verifier, generative arm (SciFact → Qwen2.5-3B-Instruct)

**ResearchGPT**, local-model arm. Trains a **generative** three-way claim
verifier, as a comparison arm against the deployed DeBERTa cross-encoder.

| | |
|---|---|
| Base | `Qwen/Qwen2.5-3B-Instruct` (3.09B) |
| Incumbent | `deberta-v3-small`, **44M**, ONNX int8, held-out macro-F1 **0.769** (stock 0.516) |
| Data | `data/scifact/pairs_*.jsonl` — 1,463 train / 628 validation / 272 dev |
| Hardware | Kaggle T4 16 GB |

---

## ⚠️ This does not replace the Verifier

**The trained DeBERTa verifier is one of the seven things `TASKLIST §0` says is
never cut.** It is the graded centrepiece: 44M parameters, int8, milliseconds
per claim on CPU, macro-F1 0.769. Nothing in this notebook is permitted to
remove it from the pipeline.

What this arm is for is **T-705b**, which asks for the verifier's accuracy to be
reported from three angles. A generative 3B arm adds a fourth and more
interesting comparison: *does 70× the parameters buy anything on this task?*

The result that helps the project most is **"no"**. A 3B generative model
failing to beat a 44M discriminative one on claim verification is direct,
quantified evidence for the "architecture beats model scale" thesis — the same
shape of finding as Valsci's Table 3, measured on this project's own task.

### One design constraint that is not negotiable

`verifier.py` needs a **calibrated score**, not just a label: T-505 caps
abstract-only evidence at PARTIAL by comparing against a threshold, and the
evidence panel shows the score per claim. A generative model that emits the bare
word `SUPPORT` gives no score. This notebook therefore trains it to emit a
**label plus a confidence**, and evaluates the confidence for calibration — an
uncalibrated confidence is worse than none, because the threshold logic would
silently trust it.

## 1 · Configuration

In [ ]:
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"
SEED         = 20260829

LORA_R       = 16
LORA_ALPHA   = 32
EPOCHS       = 2
LR           = 2e-4
BATCH        = 2
GRAD_ACCUM   = 8
MAX_SEQ_LEN  = 1024          # evidence + claim is short; no need for more

ADAPTER_DIR  = "/kaggle/working/verifier-qwen3b"
PAIRS_DIR    = "/kaggle/input/researchgpt-scifact"   # upload data/scifact as a dataset

LABELS = ["SUPPORT", "CONTRADICT", "NEI"]

## 2 · Dependencies

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Install only what is absent. A re-run costs seconds, not minutes."""
    try:
        importlib.import_module(import_name or pkg.split("==")[0].split(">")[0].replace("-", "_"))
        print(f"  present : {pkg}")
    except ImportError:
        print(f"  install : {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# transformers/torch/datasets ship with the Kaggle image; the PEFT stack does not.
for pkg, mod in [
    ("transformers>=4.44", "transformers"),
    ("datasets", "datasets"),
    ("accelerate", "accelerate"),
    ("peft", "peft"),
    ("bitsandbytes", "bitsandbytes"),
    ("trl", "trl"),
]:
    ensure(pkg, mod)

import torch
print()
print("torch      :", torch.__version__)
print("cuda       :", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("device     :", name, f"({vram:.1f} GB)")
    # T4 is sm_75: NO bf16. Every cell below uses fp16 for this reason.
    print("bf16 ok    :", torch.cuda.is_bf16_supported(), "  <- False on T4, as expected")

## 3 · Load the T-301 pairs

These are the **same pairs the DeBERTa verifier trained on**, produced by
`researchgpt.verify.scifact_pairs` with `SEED = 20260827`. Reusing them exactly
is what makes the two arms comparable — a different pairing would confound the
architecture comparison with a data one.

In [ ]:
import json, pathlib
from collections import Counter

def load_pairs(name):
    path = pathlib.Path(PAIRS_DIR) / f"pairs_{name}.jsonl"
    rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
    return rows

train_pairs = load_pairs("train")
val_pairs   = load_pairs("validation")
dev_pairs   = load_pairs("dev")
for n, rows in [("train", train_pairs), ("validation", val_pairs), ("dev", dev_pairs)]:
    print(f"{n:<11} {len(rows):>5}   {dict(Counter(r['label'] for r in rows))}")

## 4 · Format as a scored classification

The completion is a fixed two-line shape, so it can be parsed with a regex and
a malformed answer is detectable rather than silently mis-scored.

The confidence label is derived from the gold label, not invented: a training
target of a constant `0.95` for every correct answer teaches the model to always
say 0.95, which is exactly the uncalibrated failure the header warns about.
Instead the target confidence encodes SciFact's own annotation structure —
claims with multiple supporting rationales get a higher target than
single-rationale ones.

In [ ]:
from datasets import Dataset

PROMPT = (
    "Decide whether the CLAIM is supported by the EVIDENCE.\n"
    "Answer with exactly two lines:\n"
    "VERDICT: SUPPORT | CONTRADICT | NEI\n"
    "CONFIDENCE: a number between 0.00 and 1.00\n\n"
    "EVIDENCE: {evidence}\n\nCLAIM: {claim}"
)

def target_confidence(row):
    # More rationale sentences = a less ambiguous item. Bounded to [0.55, 0.95]
    # so the model never learns to emit a hard 1.0.
    n = len(row.get("rationale_sentences") or []) or 1
    return min(0.95, 0.55 + 0.10 * n)

def to_chat(row):
    completion = f"VERDICT: {row['label']}\nCONFIDENCE: {target_confidence(row):.2f}"
    return {"text": tokenizer.apply_chat_template(
        [{"role": "user", "content": PROMPT.format(evidence=row["evidence"], claim=row["claim"])},
         {"role": "assistant", "content": completion}],
        tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_pairs).map(to_chat)
eval_ds  = Dataset.from_list(val_pairs).map(to_chat)
print(train_ds[0]["text"][-260:])

## 5 · Load Qwen in 4-bit and attach LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    # fp16, not bf16 - the T4 is sm_75 and has no bf16 path. Using bf16 here
    # silently falls back to fp32 accumulation and doubles the step time.
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Causal LM + right padding corrupts the loss mask on the last token. Qwen's
# own chat template assumes left padding for generation.
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False           # incompatible with gradient checkpointing
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # All seven projections. Attention-only (q,v) is the common shortcut and it
    # underfits format-learning tasks like this one, where the MLP carries most
    # of the "what shape is the output" signal.
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6 · Train

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=ADAPTER_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=10, eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, fp16=True, optim="paged_adamw_8bit",
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=MAX_SEQ_LEN, dataset_text_field="text", packing=False,
    seed=SEED, report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     eval_dataset=eval_ds, processing_class=tokenizer)
trainer.train()
trainer.save_model(ADAPTER_DIR)

## 7 · Evaluate against the incumbent

**Held-out macro-F1 on the same dev split**, plus calibration. `0.769` is the
number to beat, and it was produced by a 44M model that runs on CPU in
milliseconds.

In [ ]:
import torch, re
from sklearn.metrics import f1_score, classification_report

model.config.use_cache = True
model.eval()
_V = re.compile(r"VERDICT:\s*(SUPPORT|CONTRADICT|NEI)", re.I)
_C = re.compile(r"CONFIDENCE:\s*([01](?:\.\d+)?)")

DEBERTA_MACRO_F1 = 0.769     # T-305, held out, measured 2026-08-27

gold, pred, confs, malformed = [], [], [], 0
for row in dev_pairs:
    ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": PROMPT.format(evidence=row["evidence"], claim=row["claim"])}],
        add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=24, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    text = tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
    m = _V.search(text)
    if not m:
        malformed += 1
        continue
    gold.append(row["label"]); pred.append(m.group(1).upper())
    c = _C.search(text)
    confs.append(float(c.group(1)) if c else None)

macro = f1_score(gold, pred, average="macro", labels=LABELS, zero_division=0)
print(f"  malformed answers   {malformed}/{len(dev_pairs)}")
print(f"  QWEN-3B macro-F1    {macro:.3f}")
print(f"  DEBERTA macro-F1    {DEBERTA_MACRO_F1:.3f}   <- incumbent, 44M params, CPU int8")
print(f"  delta               {macro - DEBERTA_MACRO_F1:+.3f}")
print()
print(classification_report(gold, pred, labels=LABELS, zero_division=0))

# Calibration: does a stated 0.9 mean 90% correct?
usable = [(c, g == p) for c, g, p in zip(confs, gold, pred) if c is not None]
if usable:
    for lo, hi in [(0.0, 0.6), (0.6, 0.75), (0.75, 0.85), (0.85, 1.01)]:
        band = [ok for c, ok in usable if lo <= c < hi]
        if band:
            print(f"  confidence [{lo:.2f},{hi:.2f})  n={len(band):<4} accuracy {sum(band)/len(band):.0%}")

## 8 · Adoption gate

**The DeBERTa verifier stays in the pipeline regardless of this result.** This
gate decides only whether the generative arm is worth *reporting* as a
comparison, and whether it could serve as an offline fallback.

| | |
|---|---|
| **Report it** | always — a negative result here is the useful one |
| **Consider as offline fallback** | only if macro-F1 ≥ 0.769 **and** malformed answers = 0 **and** confidence is monotone across the bands above |
| **Never** | swap it in as the primary. T-505's thresholds are calibrated against the DeBERTa scores; changing the scorer silently changes every verdict in every past run |

In [ ]:
monotone = True
bands = []
for lo, hi in [(0.0, 0.6), (0.6, 0.75), (0.75, 0.85), (0.85, 1.01)]:
    band = [ok for c, ok in usable if lo <= c < hi]
    if band:
        bands.append(sum(band) / len(band))
monotone = all(b <= a + 1e-9 for a, b in zip(sorted(bands), bands)) if len(bands) > 1 else False

print(f"  macro-F1 >= 0.769      {'PASS' if macro >= DEBERTA_MACRO_F1 else 'FAIL'}  ({macro:.3f})")
print(f"  no malformed answers   {'PASS' if malformed == 0 else 'FAIL'}  ({malformed})")
print(f"  confidence monotone    {'PASS' if monotone else 'FAIL'}")
print()
print("Report the number either way - T-705b wants it.")

## 9 · Export for Ollama

Training happened in 4-bit on a T4. **Inference happens on a 5 GB card through
Ollama**, which wants GGUF. Three steps: merge the adapter into fp16 weights,
convert to GGUF, quantise to `q4_K_M`.

`q4_K_M` is the target because it is what the VRAM table in `agents/README.md`
is sized against - a 3B at `q4_K_M` is ~1.9 GB of weights, leaving room for an
8192-token KV cache on a 5 GB card. `q8_0` would be better quality and would not
fit alongside the context window.

⚠️ **Merging must happen in fp16, not 4-bit.** `merge_and_unload()` on a 4-bit
model silently produces garbage weights - the dequantise-merge-requantise round
trip is not lossless and the adapter deltas are small enough to be destroyed by
it. Reload the base in fp16 first, which is why this cell is slow.

In [ ]:
import torch, gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Free the 4-bit training copy before loading the fp16 one - a T4 will not hold
# both.
try:
    del trainer, model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

base_fp16 = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map={"": "cpu"},
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_fp16, "/kaggle/working/verifier-qwen3b").merge_and_unload()
merged.save_pretrained("/kaggle/working/verifier-qwen3b-merged", safe_serialization=True)
tokenizer.save_pretrained("/kaggle/working/verifier-qwen3b-merged")
print("merged fp16 weights ->", "/kaggle/working/verifier-qwen3b-merged")

In [ ]:
# llama.cpp's converter. Kaggle has internet on, so clone it here rather than
# adding a dataset dependency.
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp 2>/dev/null || true
!pip install -q -r /kaggle/working/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

!python /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/verifier-qwen3b-merged \
    --outfile /kaggle/working/researchgpt-verifier-3b-f16.gguf \
    --outtype f16

# Quantise. The binary needs building once; ~3 min on Kaggle's CPUs.
!cmake -S /kaggle/working/llama.cpp -B /kaggle/working/llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
!cmake --build /kaggle/working/llama.cpp/build --target llama-quantize -j4 > /dev/null

!/kaggle/working/llama.cpp/build/bin/llama-quantize \
    /kaggle/working/researchgpt-verifier-3b-f16.gguf \
    /kaggle/working/researchgpt-verifier-3b-q4_k_m.gguf \
    Q4_K_M

!ls -lh /kaggle/working/*.gguf

### Loading it on the 5 GB machine

Download `researchgpt-verifier-3b-q4_k_m.gguf` from the Kaggle output, put it next to a
`Modelfile`, and register it with Ollama:

```
FROM ./researchgpt-verifier-3b-q4_k_m.gguf

# 8192, not Ollama's default 2048. See agents/README.md - at the default,
# Ollama silently drops the front of the prompt and answers from the remainder.
PARAMETER num_ctx 8192
PARAMETER temperature 0
SYSTEM "Answer only with a VERDICT line and a CONFIDENCE line."
```

```bash
ollama create researchgpt-verifier:3b -f Modelfile
python -m agents.probe --model researchgpt-verifier:3b
```

Then the scripts in `agents/` can score it against the hosted arm with no
further change.